In [ ]:
import json
from repositories import posts_repo
from utils.pagination import paginate, get_pagination_meta
from cache import posts_cache
from models.post import Post


def get_all_posts(db, page, size):
    cached = posts_cache.get_cached_posts(page, size)

    if cached:
        return json.loads(cached)

    query = posts_repo.get_all_posts(db)
    total = query.count()

    posts = paginate(query, page, size)

    result = {
        "data": [p.__dict__ for p in posts],
        "meta": get_pagination_meta(total, page, size)
    }

    posts_cache.set_cached_posts(page, size, result)
    return result


def get_post_by_id(db, post_id):
    cached = posts_cache.get_cached_post(post_id)

    if cached:
        return json.loads(cached)

    post = posts_repo.get_post_by_id(db, post_id)

    if post:
        posts_cache.set_cached_post(post_id, post.__dict__)

    return post


def create_post(db, post_data, user):
    if user.role not in ["admin", "author"]:
        raise Exception("Not allowed")

    post = posts_repo.create_post(db, post_data, user.id)

    posts_cache.invalidate_posts()

    return post


def update_post(db, post_id, update_data, user):
    post = posts_repo.get_post_by_id(db, post_id)

    if not post:
        return None

    if post.author_id != user.id and user.role != "admin":
        raise Exception("Not authorized")

    updated = posts_repo.update_post(db, post, update_data)

    posts_cache.invalidate_posts()
    posts_cache.invalidate_post(post_id)

    return updated


def delete_post(db, post_id, user):
    post = posts_repo.get_post_by_id(db, post_id)

    if not post:
        return None

    if post.author_id != user.id and user.role != "admin":
        raise Exception("Not authorized")

    posts_repo.delete_post(db, post)

    posts_cache.invalidate_posts()
    posts_cache.invalidate_post(post_id)

    return True